## Student Performance Analytics
**Tools:** Python, Pandas, NumPy, Matplotlib, Seaborn

### Objective
Explore the Kaggle *Students Performance in Exams* dataset to compare math, reading, and writing scores, inspect distributions, and investigate observed differences across test-preparation status and student groups.

### Business questions
1. What are the average, median, and spread of scores by subject?
2. How do average scores compare by test preparation status?
3. How do scores vary by gender and parental education?
4. How strongly are the subject scores associated?
5. What share of students fall into analyst-defined average-score bands?

These are descriptive comparisons and do not establish causation.


## 1. Import Libraries

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

## 2. Load and inspect data

In [ ]:
try:
    df = pd.read_csv("StudentsPerformance.csv")
except zFileNotFoundError as e:
    print(e)

In [ ]:
print("Shape:", df.shape)
display(df.head())
display(pd.DataFrame({
    "dtype" : df.dtypes.astype(str),
    "missing count" : df.isna().sum(),
    "unique_values" : df.nunique()
}))

## 3. Data quality checks

In [ ]:
score_cols = ["math score", "reading score", "writing score"]
categorical_cols = ["gender", "race/ethnicity", "parental level of education", "lunch", "test preparation course"]

print("Exact duplicate rows:", df.duplicated().sum())
print("Score summary:")
display(df[score_cols].describe().T)
for col in score_cols:
    print(f"{col}: min={df[col].min()}, max={df[col].max()}, dtype={df[col].dtypes}")

In [ ]:
for col in categorical_cols:
    # print("\\n", col)
    display(df[col].value_counts(dropna=False).to_frame("count"))

## 4. Descriptive statistics with Pandas and NumPy

In [ ]:
display(df[score_cols].agg(["count", "mean", "median", "std", "min", "max"]).T.round(2))
scores = df[score_cols].to_numpy()
# print(scores)
print("Mean of all subject-score entries:", round(np.mean(scores), 2))
print("Mean of each subject-score entries:", np.mean(scores, axis = 0).round(2))
print("Median of all subject-score entries:", round(np.median(scores), 2))
print("Standard deviation:", round(np.std(scores), 2))
print("25th, 50th, 75th percentiles:", np.percentile(scores, [25, 50, 75]))

## 5. Create analysis features

In [ ]:
analysis_df = df.copy()
analysis_df["total_score"] = df[score_cols].sum(axis = 1)
analysis_df["average_score"] = df[score_cols].mean(axis = 1).round(2)
analysis_df["score_band"] = pd.cut(
    analysis_df["average_score"],
    bins=[-float("inf"), 60, 80, float("inf")],
    labels=["Below 60", "60-79", "80-100"],
    include_lowest=True
)
display(analysis_df[score_cols + ["total_score", "average_score", "score_band"]].head())
display(analysis_df["score_band"].value_counts(sort=False).to_frame("student_count"))

## 6. Average score by subject

In [ ]:
means = df[score_cols].mean()
means.index = ["Mathematics", "Reading", "Writing"]
display(means.round(2).to_frame("Average Score"))
ax = means.plot(kind = "bar", figsize = (8, 4))
ax.set(title = "Average Score by Subject", xlabel = "Subject", ylabel = "Average Score (0-100)", ylim=(0, 100))
plt.xticks(rotation=0)
plt.tight_layout()
plt.show()

## 7. Score distributions

In [ ]:
plt.figure(figsize=(9, 5))
sns.boxplot(data=df[score_cols])
plt.title("Score Distribution by Subject")
plt.xlabel("Subject")
plt.ylabel("Score (0–100)")
plt.tight_layout()
plt.show()

## 8. Test preparation comparison

In [ ]:
test_prep_mean = df.groupby("test preparation course")[score_cols].mean().T
test_prep_mean.index = ["Mathematics", "Reading", "Writing"]
display(test_prep_mean.round(2))
ax = test_prep_mean.plot(kind = "bar", figsize=(9, 5))
ax.set(title="Average Scores by Test Preparation Status", xlabel="Subject", ylabel="Average score (0–100)", ylim=(0, 100))
plt.xticks(rotation=0)
plt.legend(title="Test preparation")
plt.tight_layout()
plt.show()

## 9. Compare average scores by gender

In [ ]:
gender_mean = df.groupby("gender")[score_cols].mean().T
gender_mean.index = ["Mathematics", "Reading", "Writing"]
display(gender_mean.round(2))
ax = gender_mean.plot(kind = "bar", figsize = (9, 5))
ax.set(title="Average Subject Scores by Gender", xlabel="Subject", ylabel="Average score (0–100)", ylim = (0, 100))
plt.xticks(rotation = 0)
plt.tight_layout()
plt.show()

## 10. Parental education comparison

In [14]:
parent_means = df.groupby("parental level of education")[score_cols].mean().sort_values("math score")
display(parent_means.round(2))
# ax = parent_means.plot(kind="barh", figsize=(10, 6))
# ax.set(title="Average Scores by Parental Education", xlabel="Average score (0–100)", ylabel="Parental education", xlim=(0, 100))
# plt.tight_layout()
# plt.show()

,math score,reading score,writing score
parental level of education,,,
high school,62.14,64.70,62.45
some high school,63.50,66.94,64.89
some college,67.13,69.46,68.84
associate's degree,67.88,70.93,69.90
bachelor's degree,69.39,73.00,73.38
master's degree,69.75,75.37,75.68


## 11. Subject score correlation

In [ ]:
corr = df[score_cols].corr()
display(corr.round(3))
plt.figure(figsize=(7, 5))
sns.heatmap(corr, annot=True, fmt=".2f", vmin=-1, vmax=1, cmap="coolwarm", square=True)
plt.title("Correlation Between Subject Scores")
plt.tight_layout()
plt.show()

## 12. Math versus reading

In [ ]:
plt.figure(figsize=(8, 6))
sns.scatterplot(data=df, x="math score", y="reading score", hue="test preparation course", alpha=0.75)
plt.title("Mathematics vs Reading Scores")
plt.tight_layout()
plt.show()

## 13. Business question summary tables

In [15]:
print("Average scores by test preparation:")
display(df.groupby("test preparation course")[score_cols].mean().round(2))
print("Average scores by parental education:")
display(df.groupby("parental level of education")[score_cols].mean().round(2))
print("Score-band distribution (%):")
display((analysis_df["score_band"].value_counts(normalize=True).sort_index() * 100).round(2).to_frame("percentage"))


Average scores by test preparation:


,math score,reading score,writing score
test preparation course,,,
completed,69.70,73.89,74.42
none,64.08,66.53,64.50


Average scores by parental education:


,math score,reading score,writing score
parental level of education,,,
associate's degree,67.88,70.93,69.90
bachelor's degree,69.39,73.00,73.38
high school,62.14,64.70,62.45
master's degree,69.75,75.37,75.68
some college,67.13,69.46,68.84
some high school,63.50,66.94,64.89


Score-band distribution (%):


,percentage
score_band,
Below 60,29.3
60-79,51.3
80-100,19.4


## 14. Findings 
- Subject with highest average:-    Reading	69.17
- Subject with lowest average:-   Mathematics	66.09
- Largest observed preparation-status difference:-    Writing	almost 10 points (completed: 74.42 && None: 64.50)
- Strongest correlation between subjects:-    Reading and Writing.
- One limitation: this is a single snapshot and group comparisons do not prove causation.


## 15. Export analysis-ready data

In [ ]:
from pathlib import Path
out_dir = Path("../Python/StudentsPerformance")
out_dir.mkdir(parents=True, exist_ok=True)
analysis_df.to_csv(out_dir / "students_performance_analysis_ready.csv", index=False)
print("Saved analysis-ready CSV.")
